# Lab 01.3 — Calcolo parallelo e distribuito

Esempi di problemi **embarrassingly parallel** (indipendenti tra loro) e degli strumenti che Spark offre per il calcolo distribuito:

1. Stima Monte Carlo di π e scalabilità al variare delle partizioni
2. Calcolo CPU-intensive distribuito (numeri primi)
3. `mapPartitions`: inizializzazione costosa una volta per partizione
4. Variabili condivise: **broadcast** e **accumulatori**
5. **Caching** dei dati riutilizzati
6. Tolleranza ai guasti: ricalcolo dal lineage

In [ ]:
import os, time, random, math, socket
from pyspark.sql import SparkSession
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("Lab01-CalcoloParallelo")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .getOrCreate())
sc = spark.sparkContext
CORES = sc.defaultParallelism
print("Master:", sc.master, "| core disponibili:", CORES)

## 1. Monte Carlo: stima di π
Lanciamo N punti casuali nel quadrato [0,1]²: la frazione che cade nel quarto di cerchio vale π/4.
Ogni partizione genera i propri punti **senza comunicare** con le altre; alla fine un `sum` aggrega i risultati.

In [ ]:
def punti_nel_cerchio(seed_and_n):
    seed, n = seed_and_n
    rnd = random.Random(seed)
    dentro = 0
    for _ in range(n):
        x, y = rnd.random(), rnd.random()
        if x * x + y * y <= 1.0:
            dentro += 1
    return dentro

def stima_pi(n_totale, n_partizioni):
    per_part = n_totale // n_partizioni
    tasks = sc.parallelize([(seed, per_part) for seed in range(n_partizioni)], n_partizioni)
    dentro = tasks.map(punti_nel_cerchio).sum()
    return 4.0 * dentro / (per_part * n_partizioni)

N = 10_000_000
t0 = time.time()
pi = stima_pi(N, CORES)
print(f"π ≈ {pi:.6f}  (errore {abs(pi - math.pi):.6f})  con {N:,} punti in {time.time() - t0:.2f}s")

### Scalabilità
Con 1 partizione lavora un solo core; aumentando le partizioni il lavoro si divide tra gli executor.
Oltre il numero di core il tempo non scende più (e piccoli task aggiungono overhead di scheduling).

In [ ]:
risultati = []
for p in [1, 2, 4, 8, 16, 64]:
    t0 = time.time()
    stima_pi(N, p)
    risultati.append((p, time.time() - t0))

base = risultati[0][1]
print("partizioni   tempo(s)   speedup")
for p, t in risultati:
    print(f"{p:10d} {t:10.2f} {base / t:9.2f}x  {'█' * int(10 * base / t)}")

In [ ]:
import matplotlib.pyplot as plt
ps, ts = zip(*risultati)
plt.figure(figsize=(6, 3))
plt.plot(ps, [base / t for t in ts], "o-", label="speedup misurato")
plt.axhline(CORES, color="gray", ls="--", label=f"core disponibili ({CORES})")
plt.xscale("log", base=2); plt.xlabel("partizioni"); plt.ylabel("speedup"); plt.legend(); plt.grid(alpha=.3)
plt.title("Monte Carlo π: speedup al variare delle partizioni"); plt.show()

## 2. Calcolo CPU-intensive: quanti numeri primi sotto 2 milioni?
Ogni partizione riceve un **intervallo** di numeri e lo verifica in autonomia.

In [ ]:
def primi_in(intervallo):
    a, b = intervallo
    def primo(n):
        if n < 2: return False
        if n % 2 == 0: return n == 2
        r = int(n ** 0.5)
        for d in range(3, r + 1, 2):
            if n % d == 0: return False
        return True
    return (socket.gethostname(), sum(1 for n in range(a, b) if primo(n)))

LIMITE, BLOCCHI = 2_000_000, 16
passo = LIMITE // BLOCCHI
intervalli = [(i * passo, (i + 1) * passo) for i in range(BLOCCHI)]

t0 = time.time()
parziali = sc.parallelize(intervalli, BLOCCHI).map(primi_in).collect()
print(f"Numeri primi < {LIMITE:,}: {sum(c for _, c in parziali):,}  in {time.time() - t0:.2f}s")
for host in sorted({h for h, _ in parziali}):
    print(f"  {host}: {sum(1 for h, _ in parziali if h == host)} blocchi elaborati")

## 3. `map` vs `mapPartitions`
Se ogni elemento richiede una risorsa costosa (connessione a DB, modello ML, ...), con `mapPartitions` la si crea **una volta per partizione** invece che una volta per elemento.

In [ ]:
def risorsa_costosa():
    time.sleep(0.01)          # simula l'apertura di una connessione
    return lambda x: x * 2

dati = sc.parallelize(range(400), 4)

t0 = time.time()
dati.map(lambda x: risorsa_costosa()(x)).count()
t_map = time.time() - t0

def per_partizione(it):
    f = risorsa_costosa()     # 1 volta per partizione
    for x in it:
        yield f(x)

t0 = time.time()
dati.mapPartitions(per_partizione).count()
t_mp = time.time() - t0
print(f"map: {t_map:.2f}s  (400 inizializzazioni)   mapPartitions: {t_mp:.2f}s  (4 inizializzazioni)")

## 4. Variabili condivise
* **Broadcast**: dato di sola lettura inviato **una volta per executor** (non una volta per task).
* **Accumulatore**: contatore in sola scrittura per i task, leggibile dal driver.

In [ ]:
# Dizionario di lookup inviato a tutti gli executor
capitali = sc.broadcast({"IT": "Roma", "FR": "Parigi", "DE": "Berlino", "ES": "Madrid"})
codici_sconosciuti = sc.accumulator(0)

def risolvi(codice):
    if codice not in capitali.value:
        codici_sconosciuti.add(1)
        return (codice, None)
    return (codice, capitali.value[codice])

codici = sc.parallelize(["IT", "FR", "XX", "DE", "IT", "YY", "ES"] * 1000, 8)
print(codici.map(risolvi).distinct().collect())
print("Codici non riconosciuti (accumulatore):", codici_sconosciuti.value)

## 5. Caching
Senza cache ogni azione ricalcola l'RDD dall'inizio del lineage. Con `cache()`/`persist()` le partizioni restano in memoria negli executor (scheda *Storage* della Spark UI).

In [ ]:
costoso = sc.parallelize(range(2_000_000), 8).map(lambda x: math.sqrt(x) * math.sin(x))

for etichetta in ["senza cache", "senza cache"]:
    t0 = time.time(); costoso.sum(); print(f"{etichetta:12s}: {time.time() - t0:.2f}s")

costoso.persist(StorageLevel.MEMORY_ONLY)
for etichetta in ["1ª (popola)", "2ª (in RAM)", "3ª (in RAM)"]:
    t0 = time.time(); costoso.sum(); print(f"{etichetta:12s}: {time.time() - t0:.2f}s")
costoso.unpersist()

## 6. Tolleranza ai guasti
Se un task fallisce, Spark lo **riesegue** (anche su un altro executor) ricostruendo la partizione dal lineage.
Simuliamo un errore al primo tentativo di una partizione: il job termina comunque correttamente.

> In modalità `local[*]` i tentativi sono disabilitati di default (`local[N, maxFailures]`): l'esempio mostra la differenza.

In [ ]:
from pyspark import TaskContext

def fallisce_al_primo_tentativo(it):
    ctx = TaskContext.get()
    if ctx.partitionId() == 0 and ctx.attemptNumber() == 0:
        raise RuntimeError("guasto simulato!")
    return [sum(it)]

try:
    print("Somma:", sum(sc.parallelize(range(100), 4).mapPartitions(fallisce_al_primo_tentativo).collect()))
    print("Il task fallito è stato rieseguito: vedi 'Failed Tasks' nella scheda Stages.")
except Exception as e:
    print("Job fallito (nessun nuovo tentativo in local mode):", type(e).__name__)

In [ ]:
spark.stop()